# USDA Cropland Data Layer (CDL) Download — Iowa

Downloads annual 30m CDL rasters for 2015–2025 from the USDA NASS release
page. Each national ZIP (~2 GB) is streamed to a temp directory, the Iowa
extent is clipped and saved, and the large national file is deleted.

Zonal statistics are then computed for each Iowa HUC12 watershed to produce
a tabular dataset of annual crop-type fractions.

**Outputs**
- `data/spatial/cdl/cdl_iowa_{year}.tif` — Iowa-clipped CDL raster per year
- `data/tabular/landuse/cdl-huc12-fractions.csv` — annual crop fractions per HUC12

**Resuming**  
Years with an existing Iowa raster are skipped in the download loop.

In [1]:
import os
import shutil
import tempfile
import zipfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterstats import zonal_stats
import requests
from pathlib import Path
from shapely.ops import unary_union

YEARS        = list(range(2015, 2026))
NASS_URL     = 'https://www.nass.usda.gov/Research_and_Science/Cropland/Release/datasets/{year}_30m_cdls.zip'
HUC12_SHP    = Path('../../data/spatial/nhdplus/wbd-huc12-iowa/WBDSnapshot_Iowa.shp')
CDL_DIR      = Path('../../data/spatial/cdl')
OUT_CSV      = Path('../../data/tabular/landuse/cdl-huc12-fractions.csv')

CDL_DIR.mkdir(parents=True, exist_ok=True)

# CDL crop code groups relevant to Iowa water quality
CROP_GROUPS = {
    'corn':        [1],
    'soybean':     [5],
    'other_crops': [2, 3, 4, 6, 21, 22, 23, 24, 25, 26, 27, 28, 29,
                    30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 41, 42,
                    43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
                    55, 56, 57, 58, 59, 61, 206, 207, 208, 209, 210,
                    211, 212, 213, 214, 215, 216, 217, 218, 219, 220,
                    221, 222, 223, 224, 225, 226, 227, 228, 229, 230,
                    231, 232, 233, 234, 235, 236, 237, 238, 239, 240,
                    241, 242, 243, 244, 245, 246, 247, 248, 249, 250,
                    251, 252, 253, 254],
    'developed':   [121, 122, 123, 124],
    'forest':      [141, 142, 143],
    'pasture':     [37, 176],
    'wetland':     [190, 195],
    'open_water':  [111],
}

print(f'Years: {YEARS[0]}–{YEARS[-1]}')
print(f'HUC12 shapefile: {HUC12_SHP}')

Years: 2015–2025
HUC12 shapefile: ../../data/spatial/nhdplus/wbd-huc12-iowa/WBDSnapshot_Iowa.shp


## 1. Load Iowa HUC12 watersheds

In [2]:
huc12 = gpd.read_file(HUC12_SHP)
print(f'HUC12 watersheds: {len(huc12)}, CRS: {huc12.crs}')

# Iowa union polygon for clipping — will be reprojected to match each CDL raster
iowa_union = unary_union(huc12.geometry)
iowa_union_gs = gpd.GeoSeries([iowa_union], crs=huc12.crs)

HUC12 watersheds: 1714, CRS: EPSG:4269


## 2. Download, clip, and save Iowa CDL rasters

In [3]:
def download_and_clip(year):
    out_tif = CDL_DIR / f'cdl_iowa_{year}.tif'
    if out_tif.exists():
        print(f'  {year}: already exists, skipping download')
        return out_tif

    url = NASS_URL.format(year=year)
    tmp_dir = Path(tempfile.mkdtemp(prefix=f'cdl_{year}_'))
    zip_path = tmp_dir / f'{year}_30m_cdls.zip'
    tif_path = tmp_dir / f'{year}_30m_cdls.tif'

    try:
        print(f'  {year}: downloading (~2 GB)...')
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(zip_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
        print(f'  {year}: extracting TIF...')
        with zipfile.ZipFile(zip_path) as z:
            tif_name = next(n for n in z.namelist() if n.endswith('.tif') and 'cdls' in n)
            z.extract(tif_name, tmp_dir)
            tif_path = tmp_dir / tif_name
        zip_path.unlink()  # free space immediately

        print(f'  {year}: clipping to Iowa...')
        with rasterio.open(tif_path) as src:
            iowa_proj = iowa_union_gs.to_crs(src.crs)
            clipped, transform = rio_mask(src, iowa_proj.geometry, crop=True)
            meta = src.meta.copy()
            meta.update({'height': clipped.shape[1], 'width': clipped.shape[2],
                         'transform': transform, 'compress': 'lzw'})
        with rasterio.open(out_tif, 'w', **meta) as dst:
            dst.write(clipped)

        size_mb = out_tif.stat().st_size / 1e6
        print(f'  {year}: saved Iowa raster ({size_mb:.0f} MB) → {out_tif}')
        return out_tif

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


for year in YEARS:
    download_and_clip(year)

print('All years downloaded and clipped.')

  2015: downloading (~2 GB)...


  2015: extracting TIF...


  2015: clipping to Iowa...


  2015: saved Iowa raster (32 MB) → ../../data/spatial/cdl/cdl_iowa_2015.tif
  2016: downloading (~2 GB)...


  2016: extracting TIF...


...

  2024: clipping to Iowa...


  2024: saved Iowa raster (30 MB) → ../../data/spatial/cdl/cdl_iowa_2024.tif
  2025: downloading (~2 GB)...


  2025: extracting TIF...


  2025: clipping to Iowa...


  2025: saved Iowa raster (34 MB) → ../../data/spatial/cdl/cdl_iowa_2025.tif
All years downloaded and clipped.


## 3. Compute zonal statistics per HUC12

In [4]:
def compute_fractions(year, huc12_gdf):
    tif = CDL_DIR / f'cdl_iowa_{year}.tif'
    with rasterio.open(tif) as src:
        raster_crs = src.crs

    huc_proj = huc12_gdf.to_crs(raster_crs)
    stats = zonal_stats(
        huc_proj,
        str(tif),
        categorical=True,
        nodata=0,
    )

    rows = []
    for feat, cat in zip(huc12_gdf.itertuples(), stats):
        if cat is None:
            continue
        total = sum(cat.values())
        if total == 0:
            continue
        row = {'year': year, 'HUC_12': feat.HUC_12}
        for group, codes in CROP_GROUPS.items():
            count = sum(cat.get(c, 0) for c in codes)
            row[f'pct_{group}'] = round(count / total, 4)
        row['pct_row_crops'] = round(
            row['pct_corn'] + row['pct_soybean'], 4
        )
        rows.append(row)
    return rows


all_rows = []
for year in YEARS:
    print(f'Computing zonal stats for {year}...')
    rows = compute_fractions(year, huc12)
    all_rows.extend(rows)
    print(f'  {year}: {len(rows)} HUC12s processed')

df = pd.DataFrame(all_rows)
df.to_csv(OUT_CSV, index=False)
print(f'\nSaved {len(df):,} rows → {OUT_CSV}')
df.head()

Computing zonal stats for 2015...


  2015: 1714 HUC12s processed
Computing zonal stats for 2016...


  2016: 1714 HUC12s processed
Computing zonal stats for 2017...


  2017: 1714 HUC12s processed
Computing zonal stats for 2018...


  2018: 1714 HUC12s processed
Computing zonal stats for 2019...


...

  2022: 1714 HUC12s processed
Computing zonal stats for 2023...


  2023: 1714 HUC12s processed
Computing zonal stats for 2024...


  2024: 1714 HUC12s processed
Computing zonal stats for 2025...


  2025: 1714 HUC12s processed

Saved 18,854 rows → ../../data/tabular/landuse/cdl-huc12-fractions.csv


,year,HUC_12,pct_corn,pct_soybean,pct_other_crops,pct_developed,pct_forest,pct_pasture,pct_wetland,pct_open_water,pct_row_crops
0,2015,102300070209,0.4761,0.3771,0.0064,0.0568,0.0121,0.0680,0.0016,0.0017,0.8532
1,2015,102300060602,0.0581,0.0476,0.0057,0.5563,0.0829,0.1233,0.0657,0.0599,0.1057
2,2015,102300050309,0.2735,0.2089,0.0074,0.0536,0.2285,0.2149,0.0071,0.0058,0.4824
3,2015,102300050308,0.4217,0.3011,0.0215,0.0678,0.0486,0.1282,0.0080,0.0030,0.7228
4,2015,102300040202,0.5165,0.3878,0.0032,0.0468,0.0020,0.0418,0.0015,0.0001,0.9043


## 4. Quick summary

In [5]:
df = pd.read_csv(OUT_CSV)
print(f'Rows: {len(df):,}  (years × HUC12s)')
print(f'Years: {sorted(df.year.unique())}')
print(f'HUC12s per year: {df.groupby("year").size().mean():.0f}')
print()
iowa_avg = df.groupby('year')[['pct_corn','pct_soybean','pct_row_crops',
                                'pct_other_crops','pct_developed',
                                'pct_forest','pct_wetland']].mean().round(3)
print('Iowa-average land cover fractions by year:')
print(iowa_avg.to_string())

Rows: 18,854  (years × HUC12s)
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
HUC12s per year: 1714

Iowa-average land cover fractions by year:
      pct_corn  pct_soybean  pct_row_crops  pct_other_crops  pct_developed  pct_forest  pct_wetland
year                                                                                               
2015     0.356        0.262          0.618            0.027          0.074       0.091        0.020
2016     0.371        0.254          0.625            0.022          0.074       0.091        0.019
2017     0.360        0.268          0.629            0.029          0.073       0.091        0.019
2018     0.355        0.273          0.628            0.032          0.073       0.089        0.022
2019     0.372        0.254          0.626            0.034          0.060       0.101        0.025
2020     0.376     